На 0.png есть только положение 0, 5, 4 метки
Предпалогаю что плитка - 3 аруко метки - 0.52 см
Правый ближний - id 0,
правая дальняя - 4,


ближняя - 5

id    coordination    смещение
0
4
5


4 (3, 0)
1 (3, 4)
2 (0, 4)

Положение камеры для метки 4: x=0.22, y=-0.12, z=0.75 м
Положение камеры для метки 2: x=0.62, y=-0.15, z=0.72 м
Положение камеры для метки 1: x=3.16, y=1.85, z=0.72 м


In [20]:
import cv2
import numpy as np

# Параметры камеры из калибровки
fx, fy = 1380.76188376, 1378.32004896
cx, cy = 985.37631507, 557.57176819
camera_matrix = np.array([[fx, 0, cx],
                          [0, fy, cy],
                          [0, 0, 1]], dtype=np.float32)
dist_coeffs = np.array([-0.00573079, -0.05853685, 0.00310127, 0.00000221, 0.0], dtype=np.float32)



marker_size = 0.07
len_brick = 0.44 # в метрах


# Координаты ArUco меток относительно центра поля (в метрах)
# {id: [x, y, z]}
marker_positions = {
    4: [3 * len_brick, 0, 0.0],   # Метка с ID 0
    2: [0, 4 * len_brick, 0.0],  # Метка с ID 4
    1: [3 * len_brick, 4 * len_brick, 0.0],  # Метка с ID 5
}



def estimate_camera_pose(corners, id, camera_matrix, dist_coeffs, marker_position, marker_size):
    marker_x, marker_y, marker_z = marker_position
    obj_points = np.array([[marker_x - marker_size / 2,  marker_y + marker_size / 2, marker_z],
                           [marker_x + marker_size / 2, marker_y + marker_size / 2, marker_z],
                           [marker_x + marker_size / 2, marker_y - marker_size / 2, marker_z],
                           [marker_x - marker_size / 2, marker_y - marker_size / 2, marker_z]], dtype=np.float32)

    # Оценка позы метки
    corners_marker = np.array(corners).reshape((4, 2))
    success, rvec, tvec = cv2.solvePnP(obj_points, corners_marker, camera_matrix, dist_coeffs)
    if not success:
        return None

    # Преобразование вектора вращения в матрицу вращения
    rmat, _ = cv2.Rodrigues(rvec)

    # Положение метки в системе координат поля
    marker_pos = np.array(marker_position, dtype=np.float32)

    # Вычисление положения камеры в системе координат метки
    camera_pos_in_marker = -np.dot(rmat.T, tvec).flatten()

    # Положение камеры в системе координат поля
    camera_pos_in_world = camera_pos_in_marker + marker_pos

    # Ориентация камеры (матрица вращения)
    camera_rotation = rmat.T
    return camera_pos_in_world, camera_rotation

def find_camera_position_by_many_markers(corners, ids, camera_matrix, dist_coeffs, marker_positions, marker_size):
    ids = list(x[0] for x in ids)
    corners = list(x[0] for x in corners)
    mark_dict = zip(ids, corners)
    marker_list = list((x) for x in mark_dict)
    for i in marker_list:
        print(i[1])
        marker_position = marker_positions[i[0]]
        result = estimate_camera_pose(i[1], i[0], camera_matrix, dist_coeffs, marker_position, marker_size)
        camera_pos, camera_rot = result
        print(f"Положение камеры для метки {i[0]}: x={camera_pos[0]:.2f}, y={camera_pos[1]:.2f}, z={camera_pos[2]:.2f} м")

def main():
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_250)
    parameters = cv2.aruco.DetectorParameters()
    detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

    frame = cv2.imread('../../datasets/test/cam0/100.png')
    frame = cv2.rotate(frame, cv2.ROTATE_180)

    corners, ids, _ = detector.detectMarkers(frame)

    # if ids is not None:
    #     cv2.aruco.drawDetectedMarkers(frame, corners)

    find_camera_position_by_many_markers(corners, ids, camera_matrix, dist_coeffs, marker_positions, marker_size)

    # Отображение кадра
    cv2.imshow('Frame', frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

[[751. 844.]
 [757. 801.]
 [830. 796.]
 [828. 839.]]
Положение камеры для метки 4: x=1.54, y=-0.12, z=0.75 м
[[376. 532.]
 [425. 528.]
 [414. 546.]
 [362. 551.]]
Положение камеры для метки 2: x=0.62, y=1.61, z=0.72 м
[[817. 503.]
 [817. 521.]
 [767. 525.]
 [768. 507.]]
Положение камеры для метки 1: x=4.48, y=3.61, z=0.72 м


In [23]:
import cv2
import numpy as np
import math

# Параметры камеры из калибровки
fx, fy = 1380.76188376, 1378.32004896
cx, cy = 985.37631507, 557.57176819
camera_matrix = np.array([[fx, 0, cx],
                          [0, fy, cy],
                          [0, 0, 1]], dtype=np.float32)
dist_coeffs = np.array([-0.00573079, -0.05853685, 0.00310127, 0.00000221, 0.0], dtype=np.float32)

# Размер метки и длина кирпича (в метрах)
marker_size = 0.07
len_brick = 0.44

# Координаты ArUco меток в мировой системе координат (в метрах)
# i : [x,y,z]
marker_positions = {
    4: [3 * len_brick, 0, 0.0],
    2: [0, 4 * len_brick, 0.0],
    1: [3 * len_brick, 4 * len_brick, 0.0],
}


def rotation_matrix_to_euler_angles(R):
    sy = math.sqrt(R[0, 0] ** 2 + R[1, 0] ** 2)
    singular = sy < 1e-6

    if not singular:
        x = math.atan2(R[2, 1], R[2, 2])
        y = math.atan2(-R[2, 0], sy)
        z = math.atan2(R[1, 0], R[0, 0])
    else:
        x = math.atan2(-R[1, 2], R[1, 1])
        y = math.atan2(-R[2, 0], sy)
        z = 0

    return np.degrees([x, y, z])  # [roll, pitch, yaw]


def find_camera_position_by_many_markers_joint(corners, ids, camera_matrix, dist_coeffs, marker_positions, marker_size):
    if ids is None:
        print("Метки не найдены.")
        return

    obj_points_all = []
    img_points_all = []

    for i, marker_id in enumerate(ids.flatten()):
        if marker_id not in marker_positions:
            continue

        marker_position = marker_positions[marker_id]
        marker_x, marker_y, marker_z = marker_position

        # 3D координаты углов метки
        obj_points = np.array([
            [marker_x - marker_size / 2, marker_y + marker_size / 2, marker_z],
            [marker_x + marker_size / 2, marker_y + marker_size / 2, marker_z],
            [marker_x + marker_size / 2, marker_y - marker_size / 2, marker_z],
            [marker_x - marker_size / 2, marker_y - marker_size / 2, marker_z]
        ], dtype=np.float32)

        img_points = corners[i].reshape((4, 2)).astype(np.float32)

        obj_points_all.append(obj_points)
        img_points_all.append(img_points)

    if not obj_points_all:
        print("Нет известных меток.")
        return

    # Объединяем все точки
    obj_points_all = np.concatenate(obj_points_all, axis=0)
    img_points_all = np.concatenate(img_points_all, axis=0)

    success, rvec, tvec = cv2.solvePnP(obj_points_all, img_points_all, camera_matrix, dist_coeffs)
    if not success:
        print("Не удалось оценить положение камеры.")
        return

    rmat, _ = cv2.Rodrigues(rvec)
    camera_pos = -np.dot(rmat.T, tvec).flatten()
    euler_angles = rotation_matrix_to_euler_angles(rmat.T)

    print(f"\n== Оценённое положение камеры ==")
    print(f"x = {camera_pos[0]:.2f} м, y = {camera_pos[1]:.2f} м, z = {camera_pos[2]:.2f} м")
    print(f"Ориентация (углы Эйлера): roll = {euler_angles[0]:.2f}°, pitch = {euler_angles[1]:.2f}°, yaw = {euler_angles[2]:.2f}°")


def main():
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_5X5_250)
    parameters = cv2.aruco.DetectorParameters()
    detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

    frame = cv2.imread('../../datasets/test/cam0/100.png')
    frame = cv2.rotate(frame, cv2.ROTATE_180)

    corners, ids, _ = detector.detectMarkers(frame)

    if ids is not None:
        cv2.aruco.drawDetectedMarkers(frame, corners, ids)

    find_camera_position_by_many_markers_joint(corners, ids, camera_matrix, dist_coeffs, marker_positions, marker_size)

    # Отображение кадра
    cv2.imshow('Frame', frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()



== Оценённое положение камеры ==
x = 1.45 м, y = -2.65 м, z = 1.95 м
Ориентация (углы Эйлера): roll = -115.87°, pitch = 0.66°, yaw = -7.45°
